In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 12.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://customize-watch-lifting.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://customize-watch-lifting.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology, MUST）是一所位於台灣新竹縣新豐鄉的私立科技大學。學校地處交通便利之處，依傍省道縱貫線並毗鄰中山高速公路，校園佔地逾三十公頃，環境優美。

**歷史沿革：**
明新科技大學的創校可追溯至1966年3月，當時以「明新工業專科學校」立案，初期設有機械、土木、工業管理三科五年制課程。 學校的創立宗旨是配合國家經濟發展，培育工業專業人才。 歷經多年的發展與擴充，於1997年7月奉教育部核准改制為「明新技術學院」並附設專科部。 最終在2002年9月，正式升格並更名為「明新科技大學」。 2018年12月，學校更名為「明新學校財團法人明新科技大學」。

**辦學理念與願景：**
學校名稱「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類與生俱來的德性與情操，培育具備高尚品德、專業學問與優良技術的全人。 明新科技大學的校訓為「堅毅、求新、創造」，並以成為「國際魅力產業科技大學」為發展願景，致力於培育「跨域整合、務實創新、全人學習」的專業人才。

**學院與學術特色：**
目前明新科技大學設有六個學院，包括：半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院。 學校涵蓋20個學系、2個學位學程（含1個博士學位學程）及11個碩士班。 值得一提的是，學校於2022年10月獲教育部核准通過「半導體科技博士學位學程」，這是其成立以來的第一個博士班，象徵著學校在學術發展上的新里程碑。 此外，明新科技大學也積極配合教育部政策，於2022年4月增設「國際專修部」，以擴大招收僑生、港澳學生及外國學生，並促進優秀人才留台就業。


In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"file","id":"615173765034672623","markAsReadToken":"hMjTVW9jFU1o3y80ZFpzQ3tgD998wDNjcq-hKTK_j6vp4HKOhxd7vLtykJIY9rFa3IwulXwbRK8GdFmIN2BzGSAQcnk99dAoYdLs4-y478p4hdQEkSrkiyoyIdDzTIemcs4_riyfR-cmqsET7Xq-MP08soKi38BFo4I7IIR_J61806akZbVDSD-AkEeFYUNteQPt7OSBz2YtSFCnjIHB6A","fileName":"作業.txt","fileSize":262,"contentProvider":{"type":"line"}},"webhookEventId":"01KS9AQA3F8VWBDTRBZSK37P92","deliveryContext":{"isRedelivery":false},"timestamp":1779503441613,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"17a8900e0b394247ab07c8bd6a0aaa25","mode":"active"}]}


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"file","id":"615173765034672623","markAsReadToken":"hMjTVW9jFU1o3y80ZFpzQ3tgD998wDNjcq-hKTK_j6vp4HKOhxd7vLtykJIY9rFa3IwulXwbRK8GdFmIN2BzGSAQcnk99dAoYdLs4-y478p4hdQEkSrkiyoyIdDzTIemcs4_riyfR-cmqsET7Xq-MP08soKi38BFo4I7IIR_J61806akZbVDSD-AkEeFYUNteQPt7OSBz2YtSFCnjIHB6A","fileName":"作業.txt","fileSize":262,"contentProvider":{"type":"line"}},"webhookEventId":"01KS9AQA3F8VWBDTRBZSK37P92","deliveryContext":{"isRedelivery":false},"timestamp":1779503441613,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"17a8900e0b394247ab07c8bd6a0aaa25","mode":"active"}]}
檔案已下載：/content/uploaded_files/作業.txt


INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:30:43] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"file","id":"615173809527849625","markAsReadToken":"ykEgfbnAP7M1q9QB87HzDzPvZM9v2xZKmx8wh7_2OURJ2zpVdrjJ8WcJpQD-DPC7Tuu61IEBA7GM01aS5ROYlOKisXN8YrxxrV5otHuFopI0Cj9k8iga3Ws54ClwL3Qt8KovCAnR_f5ltU4inNeHJp29RoWDb5Jt-Mx-QELLptNKpmZ3zaVAKYUukDga92fmzhz-eXHjg1HkfqutNgzwdg","fileName":"RAG.txt","fileSize":262,"contentProvider":{"type":"line"}},"webhookEventId":"01KS9AR3H04F43BS7VX19VGYAE","deliveryContext":{"isRedelivery":false},"timestamp":1779503468046,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"1c7779c05c2d420db4be3820c5f242b8","mode":"active"}]}


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"file","id":"615173809527849625","markAsReadToken":"ykEgfbnAP7M1q9QB87HzDzPvZM9v2xZKmx8wh7_2OURJ2zpVdrjJ8WcJpQD-DPC7Tuu61IEBA7GM01aS5ROYlOKisXN8YrxxrV5otHuFopI0Cj9k8iga3Ws54ClwL3Qt8KovCAnR_f5ltU4inNeHJp29RoWDb5Jt-Mx-QELLptNKpmZ3zaVAKYUukDga92fmzhz-eXHjg1HkfqutNgzwdg","fileName":"RAG.txt","fileSize":262,"contentProvider":{"type":"line"}},"webhookEventId":"01KS9AR3H04F43BS7VX19VGYAE","deliveryContext":{"isRedelivery":false},"timestamp":1779503468046,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"1c7779c05c2d420db4be3820c5f242b8","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/bkrcefpzc7uz


INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:31:11] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"text","id":"615173825768194323","quoteToken":"95hXAoNaawgrpdvBg89ZME6cOopSGxBNfwHmEZDxxDMudDjnSx92fPVQg_83OaGUP4J_22IGeXspmfkXmN9QNnScPixrQbfGjHD3flUIfGwlcVXCapVtOWONR3SqEfWZ1O0vHqOVqvAK26js-7Urzg","markAsReadToken":"SRdDh_bga1tF-pBrvdaBs7OU7Z5VA4XN8fq0BOkzTklzfzyJR8MIW-Yj_DQBBE8nSNXOkFJVePmddKCgj2x_akG81OAVgLNxDF-jFF58O4AhP66b7ejv3_hnSRskJeF5hVk7Zk9NohOKYq50WAKCwVWSfXxYxh0OJPbelK-mxr8KtOXDAdQ1sBPlzC08IFfT1JeGFtLNdkoz5oM5TwUAIQ","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KS9ARDH68VGRWRG7W667J4NF","deliveryContext":{"isRedelivery":false},"timestamp":1779503477803,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"ec290550cb5141d4abbf03f3dd5a37c7","mode":"active"}]}


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"text","id":"615173825768194323","quoteToken":"95hXAoNaawgrpdvBg89ZME6cOopSGxBNfwHmEZDxxDMudDjnSx92fPVQg_83OaGUP4J_22IGeXspmfkXmN9QNnScPixrQbfGjHD3flUIfGwlcVXCapVtOWONR3SqEfWZ1O0vHqOVqvAK26js-7Urzg","markAsReadToken":"SRdDh_bga1tF-pBrvdaBs7OU7Z5VA4XN8fq0BOkzTklzfzyJR8MIW-Yj_DQBBE8nSNXOkFJVePmddKCgj2x_akG81OAVgLNxDF-jFF58O4AhP66b7ejv3_hnSRskJeF5hVk7Zk9NohOKYq50WAKCwVWSfXxYxh0OJPbelK-mxr8KtOXDAdQ1sBPlzC08IFfT1JeGFtLNdkoz5oM5TwUAIQ","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KS9ARDH68VGRWRG7W667J4NF","deliveryContext":{"isRedelivery":false},"timestamp":1779503477803,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"ec290550cb5141d4abbf03f3dd5a37c7","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:31:20] "POST / HTTP/1.1" 200 -
